In [ ]:
!pip install -q marker-pdf pymupdf

In [ ]:
# ─────────────────────────────────────────────
# CONFIGURACIÓN
# ─────────────────────────────────────────────
# INPUT_PDF  → str con ruta a un PDF, lista de rutas, o None para usar INPUT_DIR
# INPUT_DIR  → carpeta con varios PDFs (solo si INPUT_PDF es None)
# OUTPUT_DIR → carpeta donde se guardarán los archivos Markdown

INPUT_PDF  = "/Users/ldiaz/Documents/Documentos/otros/thesis_apply/fujita2009.pdf"
INPUT_DIR  = "/kaggle/input/pdf"
OUTPUT_DIR = "/Users/ldiaz/Documents/Documentos/otros/thesis_apply/.temp"

In [ ]:
import re
import sys
from pathlib import Path
from datetime import datetime


# ─────────────────────────────────────────────
# EXTRACCIÓN CON MARKER-PDF
# ─────────────────────────────────────────────

def convertir_con_marker(pdf_path: Path, output_dir: Path) -> dict:
    try:
        from marker.converters.pdf import PdfConverter
        from marker.models import create_model_dict
        from marker.output import text_from_rendered
        from marker.config.parser import ConfigParser

        config = ConfigParser({"output_format": "markdown", "langs": "es,en"})
        models = create_model_dict()
        converter = PdfConverter(
            config=config.generate_config_dict(),
            artifact_dict=models,
            processor_list=config.get_processors(),
            renderer=config.get_renderer(),
        )
        rendered = converter(str(pdf_path))
        full_text, _, images = text_from_rendered(rendered)
        return {"markdown": full_text, "images": images, "metadata": {}, "method": "marker"}

    except ImportError:
        print("  [!] marker-pdf no encontrado. Usando PyMuPDF como fallback.")
        return convertir_con_pymupdf(pdf_path, output_dir)
    except Exception as e:
        print(f"  [!] marker falló: {e}. Usando PyMuPDF como fallback.")
        return convertir_con_pymupdf(pdf_path, output_dir)


# ─────────────────────────────────────────────
# FALLBACK: EXTRACCIÓN CON PYMUPDF
# ─────────────────────────────────────────────

def convertir_con_pymupdf(pdf_path: Path, output_dir: Path) -> dict:
    import fitz

    doc = fitz.open(str(pdf_path))
    md_parts = []
    images = {}

    for page_num, page in enumerate(doc, start=1):
        blocks = page.get_text("dict")["blocks"]
        for block in blocks:
            if block["type"] == 0:
                for line in block["lines"]:
                    text = " ".join(span["text"] for span in line["spans"]).strip()
                    if text:
                        font_size = line["spans"][0]["size"] if line["spans"] else 12
                        if font_size > 14:
                            md_parts.append(f"\n## {text}\n")
                        else:
                            md_parts.append(text)
            elif block["type"] == 1:
                img_index = len(images)
                img_name = f"fig_{page_num}_{img_index}.png"
                try:
                    xref = block.get("xref") or block.get("image", {}).get("xref")
                    if xref:
                        base_image = doc.extract_image(xref)
                        images[img_name] = base_image["image"]
                        md_parts.append(f"\n![Figura {img_index + 1}](images/{img_name})\n")
                except Exception:
                    pass
        md_parts.append("\n\n---\n\n")

    doc.close()
    return {"markdown": "\n".join(md_parts), "images": images, "metadata": {}, "method": "pymupdf"}


# ─────────────────────────────────────────────
# UTILIDADES
# ─────────────────────────────────────────────

def limpiar_nombre(nombre: str) -> str:
    nombre = nombre.lower()
    for src, dst in [("áàä","a"),("éèë","e"),("íìï","i"),("óòö","o"),("úùü","u"),("ñ","n")]:
        for c in src:
            nombre = nombre.replace(c, dst)
    nombre = re.sub(r"[^a-z0-9_\-]", "_", nombre)
    nombre = re.sub(r"_+", "_", nombre).strip("_")
    return nombre[:80]

def contar_secciones(markdown: str) -> int:
    return len(re.findall(r"^#{1,3} .+", markdown, re.MULTILINE))


# ─────────────────────────────────────────────
# GUARDAR RESULTADO
# ─────────────────────────────────────────────

def guardar_articulo(resultado: dict, pdf_path: Path, output_dir: Path) -> Path:
    import io
    nombre_base = limpiar_nombre(pdf_path.stem)
    carpeta = output_dir / nombre_base
    carpeta.mkdir(parents=True, exist_ok=True)

    md_file = carpeta / f"{nombre_base}.md"
    header = f"""---
archivo_original: {pdf_path.name}
convertido: {datetime.now().strftime('%Y-%m-%d %H:%M')}
metodo: {resultado.get('method', 'desconocido')}
---

"""
    md_file.write_text(header + resultado["markdown"], encoding="utf-8")

    images = resultado.get("images", {})
    if images:
        images_dir = carpeta / "images"
        images_dir.mkdir(exist_ok=True)
        for img_name, img_data in images.items():
            img_path = images_dir / img_name
            try:
                if isinstance(img_data, bytes):
                    img_path.write_bytes(img_data)
                elif hasattr(img_data, "save"):
                    buf = io.BytesIO()
                    img_data.convert("RGB").save(buf, format="JPEG")
                    img_path.with_suffix(".jpg").write_bytes(buf.getvalue())
            except Exception as e:
                print(f"    [!] No se pudo guardar {img_name}: {e}")

    print(f"  ✓ {carpeta.name}/ ({len(images)} imágenes) → {md_file}")
    return md_file


# ─────────────────────────────────────────────
# ÍNDICE
# ─────────────────────────────────────────────

def generar_indice(articulos: list, output_dir: Path):
    lines = [
        "# Índice de Artículos Científicos\n",
        f"*Generado: {datetime.now().strftime('%Y-%m-%d %H:%M')} — {len(articulos)} artículos*\n",
        "\n| # | Archivo | Secciones | Imágenes | Estado |\n",
        "|---|---------|-----------|----------|--------|\n",
    ]
    for i, art in enumerate(articulos, start=1):
        nombre = art["nombre"]
        estado = "✅" if art.get("ok") else "❌"
        md_link = f"[{nombre}](./{nombre}/{nombre}.md)"
        lines.append(f"| {i} | {md_link} | {art.get('secciones',0)} | {art.get('imagenes',0)} | {estado} |\n")

    indice_path = output_dir / "_indice.md"
    indice_path.write_text("".join(lines), encoding="utf-8")
    print(f"\n📋 Índice: {indice_path}")

print("Funciones cargadas.")

In [ ]:
# ─────────────────────────────────────────────
# EJECUCIÓN PRINCIPAL
# ─────────────────────────────────────────────

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# Determinar lista de PDFs a procesar
if INPUT_PDF is None:
    pdfs = sorted(Path(INPUT_DIR).glob("*.pdf"))
elif isinstance(INPUT_PDF, list):
    pdfs = [Path(p) for p in INPUT_PDF]
else:
    pdfs = [Path(INPUT_PDF)]

if not pdfs:
    raise FileNotFoundError(f"No se encontraron PDFs en: {INPUT_DIR}")

print(f"🔬 PDFs a procesar: {len(pdfs)}")
print(f"📁 Salida: {output_dir}\n")

articulos = []
errores = []

for idx, pdf_path in enumerate(pdfs, start=1):
    nombre_base = limpiar_nombre(pdf_path.stem)
    destino = output_dir / nombre_base / f"{nombre_base}.md"

    print(f"[{idx:03d}/{len(pdfs)}] {pdf_path.name}")

    if destino.exists():
        print(f"  ⏭  Ya existe, omitiendo")
        md_text = destino.read_text(encoding="utf-8")
        imgs = list((output_dir / nombre_base / "images").glob("*")) if (output_dir / nombre_base / "images").exists() else []
        articulos.append({"nombre": nombre_base, "secciones": contar_secciones(md_text), "imagenes": len(imgs), "ok": True})
        continue

    try:
        resultado = convertir_con_marker(pdf_path, output_dir)
        md_file = guardar_articulo(resultado, pdf_path, output_dir)
        md_text = md_file.read_text(encoding="utf-8")
        articulos.append({
            "nombre": nombre_base,
            "secciones": contar_secciones(md_text),
            "imagenes": len(resultado.get("images", {})),
            "ok": True,
        })
    except Exception as e:
        print(f"  ❌ Error: {e}")
        errores.append({"pdf": pdf_path.name, "error": str(e)})
        articulos.append({"nombre": nombre_base, "secciones": 0, "imagenes": 0, "ok": False})

generar_indice(articulos, output_dir)

exitosos = sum(1 for a in articulos if a["ok"])
print(f"\n{'─'*50}")
print(f"✅ Completados : {exitosos}/{len(pdfs)}")
if errores:
    print(f"❌ Con errores : {len(errores)}")
    for err in errores:
        print(f"   - {err['pdf']}: {err['error']}")
print(f"📁 Resultado en: {output_dir.resolve()}")

In [ ]:
# !rm -rf articulos/